In [ ]:
cache_path = '/data/guidance-team-new/dnncache/'
import os
os.environ['DNNLIB_CACHE_DIR'] = cache_path

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import requests
import os
import tqdm
import sys
import base.dnnlib as dnnlib
import pickle

import matplotlib.pyplot as plt

from cl_9_openai import create_classifier

sys.path.append('./base')
# ======== 1. 載入模型 ========
# classifier = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
# classifier = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
device='cuda:0'
model_path = '/data/guidance-team-new/classifier_log_trained_by_train_res50_9/checkpoint_epoch17_batch180000.pt'

model = create_classifier(1000,4, True)
# model = create_latent_convnext_ht_classifier(args.num_classes, args.in_channels, args.pretrained)
model.to(device)
print(f"Loading checkpoint from {model_path}...")
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
classifier = model
classifier.eval()

# ======== 2. 定義預處理 ========
# preprocess = transforms.Compose([
#     transforms.Resize(256),
#     transforms.CenterCrop(224),
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     )
# ])
weights = models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
preprocess = None
# preprocess = transforms.Compose([
#     transforms.Resize(256),   # 先放大到 256x256
#     transforms.RandomResizedCrop(224),
#     transforms.RandomHorizontalFlip(),
#     transforms.ColorJitter(brightness=0.2, contrast=0.2),
#     transforms.RandomRotation(degrees=15),
    
#     transforms.ToTensor(),
#     transforms.Normalize(
#         mean=[0.485, 0.456, 0.406],
#         std=[0.229, 0.224, 0.225]
#     ),
    
#     # RandomErasing 應該放在 ToTensor 和 Normalize 之後
#     transforms.RandomErasing(p=0.2, scale=(0.02, 0.33)), 
# ])
# ======== 3. 建立資料集和 DataLoader ========
# 確保 training.dataset 模組和 dnnlib 已被正確引入

data_loader_kwargs = dict(
    class_name='torch.utils.data.DataLoader',
    pin_memory=True,
    num_workers=2,
    prefetch_factor=2
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


val_data_dir = '/data/guidance-team-new/Imagenet/val_latent/'
valid_64_dataset_kwargs = dict(
    class_name='training.dataset.ImageFolderDataset',
    path=val_data_dir
)
valid_64_dataset_obj = dnnlib.util.construct_class_by_name(**valid_64_dataset_kwargs)

val_64_data_loader = dnnlib.util.construct_class_by_name(
    dataset=valid_64_dataset_obj,
    batch_size=32,
    **data_loader_kwargs
)


# ======== 4. 載入 ImageNet 標籤 ========
# 只需要載入一次標籤
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.strip().split("\n")

# ======== 4. Diffusion model ========
# net = 'https://nvlabs-fi-cdn.nvidia.com/edm2/posthoc-reconstructions/edm2-img64-s-1073741-0.075.pkl'
net = 'https://nvlabs-fi-cdn.nvidia.com/edm2/posthoc-reconstructions/edm2-img512-s-2147483-0.085.pkl'
if isinstance(net, str):
    print(f'Loading main network from {net} ...')
    with dnnlib.util.open_url(net, verbose=True) as f:
        data = pickle.load(f)
    net = data['ema'].to(device)
    encoder = data.get('encoder', None)
    if encoder is None:
        encoder = dnnlib.util.construct_class_by_name(class_name='training.encoders.StandardRGBEncoder')
assert net is not None
num_steps = 32
test_t_list = [0.002]
num_steps = 32
sigma_max = 80
sigma_min = 0.002
rho = 7

step_indices = torch.arange(num_steps)

t_steps = (
    sigma_max ** (1 / rho)
    + step_indices / (num_steps - 1)
      * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
) ** rho

# 最後一個 step 設為 0
t_steps = torch.cat([t_steps, torch.zeros_like(t_steps[:1])])
accuracy_list = []
for t in reversed(t_steps[:-1:2]):
    test_t_list.append(t)
print(f'test_t_list:{test_t_list}')
for idx,t_val in enumerate(test_t_list):        
    # ======== 5. 執行驗證迴圈 ========
    classifier.to(device)
    correct_predictions = 0
    total_samples = 0
    examples_shown = False # 新增一個旗標來確保只印出一次範例
    
    with torch.no_grad():
        for i, (images_meanstd, labels_idx) in enumerate(tqdm.tqdm(val_64_data_loader, total=len(val_64_data_loader))):
            # 將資料移到 GPU
            images_meanstd = images_meanstd.to(device)
            labels_idx = labels_idx.to(device)
            images_latent = encoder.encode_latents(images_meanstd.to(device))
    
            t_to_test = torch.full((images_latent.shape[0],), t_val, device=device)  # 建立所有值都為 t_val 的 tensor
            xt = images_latent+ torch.randn_like(images_latent) * t_to_test.view(-1, 1, 1, 1)
            xo_predicted = net(xt,t_to_test,labels_idx)
            # 從 64x64 的張量轉換為 224x224
            # 1. 將張量從 CUDA 移到 CPU，並將其範圍從 [-1, 1] 轉換到 [0, 1]
            # xo_predicted_cpu = (xo_predicted.cpu() * 0.5 + 0.5).clamp(0, 1)
            dummy_t = torch.ones(labels_idx.shape[0]).to(device)
    
            # 2. 使用 torchvision 的 transforms 對每張圖片進行處理
            # images_224 = torch.stack([preprocess(transforms.ToPILImage()(img)) for img in xo_predicted_cpu])
            
            # 3. 將處理後的張量移回 CUDA 裝置
            # images_224 = images_224.to(device)
    
            # logits = classifier(xo_predicted)
            outputs = classifier(xo_predicted,dummy_t)
            if labels_idx.dim() > 1:
                labels_idx = torch.argmax(labels_idx, dim=1)
            
            labels_idx = labels_idx.to(device)
    
            # 進行預測
            # outputs = classifier(images)
            _, predicted_labels = torch.max(outputs, 1)
    
            # 計算正確預測數
            correct_predictions += (predicted_labels == labels_idx).sum().item()
            total_samples += labels_idx.size(0)
    
            # --- 顯示第一個批次的範例 ---
            if i == 0 and not examples_shown:
                print("\n--- 顯示第一個批次的範例預測 ---")
                
                # 從這個批次中選擇 3 張圖片來顯示
                for j in range(1):
                    # 取得單張圖片、預測標籤和真實標籤
                    # single_image = images_224[j].cpu()
                    single_latent = xo_predicted[0]
                    single_image = encoder.decode(single_latent)[0]
                    single_output = outputs[j].cpu()
                    true_label_idx = labels_idx[j].cpu().item()
                    
                    # 反正規化以顯示圖片
                    # 這是預處理的反向操作
                    # unnormalize = transforms.Normalize(
                    #     mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
                    #     std=[1/0.229, 1/0.224, 1/0.225]
                    
                    # display_image = unnormalize(single_image)
                    print('minmax of single_image',single_image.min(),single_image.max())
                    display_image = single_image.cpu()
                    # 將 CHW 轉為 HWC，並將 Tensor 轉為 numpy array
                    display_image = display_image.permute(1, 2, 0).numpy()
                    display_image = (display_image).astype('uint8') # 轉為 uint8 格式
    
                    # 取得 Top-5 預測
                    probs = torch.nn.functional.softmax(single_output, dim=0)
                    top5_prob, top5_catid = torch.topk(probs, 5)
    
                    # 顯示圖片和標題
                    plt.figure()
                    plt.imshow(display_image)
                    plt.axis("off")
                    
                    title = f"True: {labels[true_label_idx]}\n"
                    title += "Top-5 Predictions:\n"
                    for k in range(top5_prob.size(0)):
                        # print(f'labels:{labels}')
                        title += f"  {labels[top5_catid[k]]}: {top5_prob[k].item():.2%}\n"
                    
                    plt.title(title, fontsize=10, color='blue')
                    plt.show()
    
                examples_shown = True
    
    # ======== 6. 計算並列印準確率 ========
    accuracy = correct_predictions / total_samples
    accuracy_list.append(accuracy)
    print(f"t=: {t_val}")
    print(f"總共處理了 {total_samples} 張圖片。")
    print(f"正確預測了 {correct_predictions} 張圖片。")
    print(f"模型在驗證集上的準確率為: {accuracy:.4f}")


import datetime
import pickle


timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# 創建檔案名稱，將時間戳加入其中
file_path = f"./classifier_test_result/{timestamp}.pkl"

# 保存列表到檔案
with open(file_path, 'wb') as f:
    pickle.dump(accuracy_list, f)

print(f"列表已成功保存到 {file_path}")
x_values = [i * 2 for i in range(len(accuracy_list))]

plt.plot(x_values,accuracy_list)

# 添加標題和軸標籤
plt.title("Accuracy of x0 estimation")
plt.xlabel("difufsion steps")
plt.ylabel("Accuracy")

# 顯示圖表
plt.show()

In [ ]:
import pickle

file_path = f"./classifier_test_result/res50v2_nofinetune.pkl"

with open(file_path, 'rb') as f:
    accuracy_list_loaded = pickle.load(f)

x_values = [i * 2 for i in range(len(accuracy_list))]

plt.plot(x_values,accuracy_list_loaded)

# 添加標題和軸標籤
plt.title("Accuracy of x0 estimation")
plt.xlabel("difufsion steps")
plt.ylabel("Accuracy")

# 顯示圖表
plt.show()

test_t_list_integer = [int(t) for t in test_t_list]

plt.plot(test_t_list_integer,accuracy_list_loaded)

# 添加標題和軸標籤
plt.title("Accuracy of x0 estimation")
plt.xlabel("sigma")
plt.ylabel("Accuracy")

# 顯示圖表
plt.show()

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import requests
import os
import tqdm
import sys
import base.dnnlib as dnnlib
import pickle

import numpy as np
import matplotlib.pyplot as plt
import random
import torchvision.transforms.functional as F
import torch.nn.functional as Fnn


sys.path.append('./base')

classifier = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
classifier.eval()

data_loader_kwargs = dict(
    class_name='torch.utils.data.DataLoader',
    pin_memory=True,
    num_workers=2,
    prefetch_factor=2
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


val_data_dir = '/data/guidance-team-new/Imagenet/val_latent/'
valid_64_dataset_kwargs = dict(
    class_name='training.dataset.ImageFolderDataset',
    path=val_data_dir
)
valid_64_dataset_obj = dnnlib.util.construct_class_by_name(**valid_64_dataset_kwargs)

val_64_data_loader = dnnlib.util.construct_class_by_name(
    dataset=valid_64_dataset_obj,
    batch_size=32,
    **data_loader_kwargs
)


# ======== 4. 載入 ImageNet 標籤 ========
# 只需要載入一次標籤
LABELS_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
labels = requests.get(LABELS_URL).text.strip().split("\n")

# ======== 4. Diffusion model ========
net = 'https://nvlabs-fi-cdn.nvidia.com/edm2/posthoc-reconstructions/edm2-img512-s-2147483-0.085.pkl'
if isinstance(net, str):
    print(f'Loading main network from {net} ...')
    with dnnlib.util.open_url(net, verbose=True) as f:
        data = pickle.load(f)
    net = data['ema'].to(device)
    encoder = data.get('encoder', None)
    if encoder is None:
        encoder = dnnlib.util.construct_class_by_name(class_name='training.encoders.StandardRGBEncoder')
assert net is not None
num_steps = 32
test_t_list = [0.002]
num_steps = 32
sigma_max = 80
sigma_min = 0.002
rho = 7

step_indices = torch.arange(num_steps)

t_steps = (
    sigma_max ** (1 / rho)
    + step_indices / (num_steps - 1)
      * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
) ** rho

# 最後一個 step 設為 0
t_steps = torch.cat([t_steps, torch.zeros_like(t_steps[:1])])



accuracy_list = []
for t in reversed(t_steps[:-1:2]):
    test_t_list.append(t)
print(f'test_t_list:{test_t_list}')
for idx,t_val in enumerate(test_t_list):        
    # ======== 5. 執行驗證迴圈 ========
    classifier.to(device)
    correct_predictions = 0
    total_samples = 0
    examples_shown = False # 新增一個旗標來確保只印出一次範例
    
    with torch.no_grad():
        for i, (images_meanstd, labels_idx) in enumerate(tqdm.tqdm(val_64_data_loader, total=len(val_64_data_loader))):
            # 將資料移到 GPU
            images_meanstd = images_meanstd.to(device)
            labels_idx = labels_idx.to(device)
            images_latent = encoder.encode_latents(images_meanstd)
            images_latent_flip = torch.flip(images_latent, dims=[-1])
            images_latent_rotate = torch.rot90(images_latent, k=1, dims=[-2, -1]) # 90 degrees
            images_latent_rotate10 = F.rotate(images_latent, angle=10) # <-- 這裡
            images_latent_mask0 = images_latent.clone()
            images_latent_mask0[:, 0, :, :] = 0
            images_latent_mask1 = images_latent.clone()
            images_latent_mask1[:, 1, :, :] = 0
            images_latent_mask2 = images_latent.clone()
            images_latent_mask2[:, 2, :, :] = 0
            images_latent_mask3 = images_latent.clone()
            images_latent_mask3[:, 3, :, :] = 0
            _, _, h, w = images_latent.shape
            crop_size = random.randint(int(h * 0.8), h) # 隨機裁剪 80% 到 100% 的區域
            start_y = random.randint(0, h - crop_size)
            start_x = random.randint(0, w - crop_size)
            cropped_tensor = images_latent[:, :, start_y:start_y+crop_size, start_x:start_x+crop_size]
            images_latent_crop = Fnn.interpolate(cropped_tensor, size=(64, 64), mode='bilinear', align_corners=False)
            
            images_512 = encoder.decode(images_latent.to(device))
            images_512_flip = encoder.decode(images_latent_flip.to(device))
            images_512_rotate = encoder.decode(images_latent_rotate.to(device))
            images_512_rotate10 = encoder.decode(images_latent_rotate10.to(device))
            images_512_mask0 = encoder.decode(images_latent_mask0.to(device))
            images_512_mask1 = encoder.decode(images_latent_mask1.to(device))
            images_512_mask2 = encoder.decode(images_latent_mask2.to(device))
            images_512_mask3 = encoder.decode(images_latent_mask3.to(device))
            images_512_crop = encoder.decode(images_latent_crop.to(device))
            # 2. 使用 torchvision 的 transforms 對每張圖片進行處理
            # images_512 = torch.stack([preprocess(transforms.ToPILImage()(img)) for img in xo_predicted_cpu])
    
                    # 取得單張圖片、預測標籤和真實標籤
            if labels_idx.dim() > 1:
                labels_idx = torch.argmax(labels_idx, dim=1)
            # print("\n--- images_512 資訊 ---")
            # print(f"型態 (Type): {images_512.dtype}")
            # print(f"裝置 (Device): {images_512.device}")
            # print(f"大小 (Shape): {images_512.shape}")
            # print(f"最大值 (Max value): {images_512.max().item():.4f}")
            # print(f"最小值 (Min value): {images_512.min().item():.4f}")
            # print("--- 檢查結束 ---\n")
            # single_image = images_512[0].cpu()
            # true_label_idx = labels_idx[0].cpu().item()
            # display_image = single_image.permute(1, 2, 0).numpy()
            # # 顯示圖片和標題
            # plt.figure()
            # plt.imshow(display_image)
            # plt.axis("off")
            
            # title = f"True: {labels[true_label_idx]}\n"
            # plt.title(title, fontsize=10, color='blue')
            # plt.show()
            # break
            for idx in range(images_512.shape[0]):
                images_to_display = [
                    (images_512[idx].cpu(), "Original"),
                    (images_512_flip[idx].cpu(), "Flipped"),
                    (images_512_rotate[idx].cpu(), "Rot 90"),
                    (images_512_rotate10[idx].cpu(),"Rot 10"),
                    (images_512_mask0[idx].cpu(),"Mask 0"),
                    (images_512_mask1[idx].cpu(),"Mask 1"),
                    (images_512_mask2[idx].cpu(),"Mask 2"),
                    (images_512_mask3[idx].cpu(),"Mask 3"),
                    (images_512_crop[idx].cpu(),"Crop")
                ]
                
                # 取得真實標籤
                true_label_idx = labels_idx[0].cpu().item()
                title_label = labels[true_label_idx]
                
                # 設定 Matplotlib 繪圖
                # 根據你的圖片數量動態調整子圖大小，一行最多放 4 張
                num_images = len(images_to_display)
                cols = 3
                rows = (num_images + cols - 1) // cols  # 計算所需的行數
                
                fig, axes = plt.subplots(rows, cols, figsize=(20, 5 * rows))
                
                # 處理 axes 可能是一維或二維陣列的情況
                if rows > 1:
                    axes = axes.flatten()
                else:
                    # 當只有一行時，axes 是一個一維陣列，不需要 flatten
                    pass
                    
                for i, (decoded_image, title) in enumerate(images_to_display):
                    # 將張量從 [0, 255] 轉換為適合顯示的格式
                    # decoded_image.cpu() 已經在列表定義時完成，這裡無需再次呼叫
                    display_image = decoded_image.permute(1, 2, 0).numpy().astype(np.uint8)
                    
                    # 顯示圖片
                    ax = axes[i]
                    ax.imshow(display_image)
                    ax.set_title(f"{title}\nTrue: {title_label}", fontsize=10, color='blue')
                    ax.axis('off')
                
                # 隱藏多餘的子圖（如果有的話）
                for i in range(num_images, len(axes)):
                    axes[i].axis('off')
                
                plt.tight_layout()
                plt.show()
            break
    break